
# 🧪 TF‑IDF vs. Embeddings vs. Hybrid with XGBoost (20 Newsgroups)

This notebook runs three variants side-by-side on the **20 Newsgroups** dataset:

1. **TF‑IDF → XGBoost**  
2. **Sentence Embeddings (all-MiniLM-L6-v2) → XGBoost**  
3. **Hybrid (TF‑IDF ⊕ Embeddings) → XGBoost**  

It performs k-fold CV, trains final models, and produces a comparison table.

**How to use (Colab):**
1. Open in Colab (or run locally).  
2. (Optional) Enable GPU in Colab: `Runtime → Change runtime type → GPU`.  
3. Run cells top-to-bottom.  
4. Inspect the final results table (Macro-F1, Accuracy, ROC-AUC).

**Notes**
- Primary metric is **Macro-F1** (robust to class imbalance).  
- We keep vectorizer sizes modest for speed.  
- You can tweak grids / parameters in one place.  


In [ ]:

# If running in Colab, uncomment the next line
# !pip -q install xgboost scikit-learn sentence-transformers scipy pandas numpy


In [ ]:

import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_20newsgroups
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from scipy.sparse import csr_matrix, hstack

import time, sys, platform, math, os
SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5
np.random.seed(SEED)

def detect_tree_method():
    # Use GPU if available (works in Colab if GPU runtime is on)
    try:
        import xgboost as xgb
        if hasattr(xgb, 'core') and hasattr(xgb, 'rabit'):
            # Heuristic: respect COLAB_GPU env or CUDA presence
            has_cuda = os.path.exists('/proc/driver/nvidia/version')
            return 'gpu_hist' if has_cuda else 'hist'
    except Exception:
        pass
    return 'hist'

TREE_METHOD = detect_tree_method()
TREE_METHOD


In [ ]:

# Load 20 Newsgroups (remove headers/quotes to make it a bit more content-based)
newsgroups = fetch_20newsgroups(subset='all', remove=('headers','quotes'))
texts = newsgroups.data
labels = newsgroups.target
target_names = newsgroups.target_names

print(f"Samples: {len(texts)}, Classes: {len(target_names)}")
print("Example label names:", target_names[:5])

# Train-test split
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)

num_classes = len(np.unique(labels))
num_classes


In [ ]:

def evaluate_clf(clf, X, y):
    y_pred = clf.predict(X)
    f1 = f1_score(y, y_pred, average='macro')
    acc = accuracy_score(y, y_pred)
    roc = np.nan
    try:
        proba = clf.predict_proba(X)
        if len(np.unique(y)) == 2:
            roc = roc_auc_score(y, proba[:,1])
        else:
            roc = roc_auc_score(y, proba, multi_class='ovr')
    except Exception:
        pass
    return {'macro_f1': f1, 'accuracy': acc, 'roc_auc': roc}

def cv_run(X, y, build_model_fn, n_splits=N_SPLITS, seed=SEED, verbose=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics = []
    start = time.time()
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = build_model_fn()
        clf.fit(X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric='mlogloss' if len(np.unique(y)) > 2 else 'logloss',
                verbose=False,
                early_stopping_rounds=50)
        fold_metrics = evaluate_clf(clf, X_va, y_va)
        metrics.append(fold_metrics)
        if verbose:
            print(f"Fold {fold}: ", fold_metrics)
    dur = time.time() - start
    df = pd.DataFrame(metrics)
    return df.mean().to_dict(), df.std().to_dict(), dur


In [ ]:

# ---------- Variant A: TF-IDF ----------
def build_tfidf_features(train_texts, test_texts,
                         ngram_range=(1,2),
                         max_features=30000, min_df=5, max_df=0.9,
                         sublinear_tf=True, norm='l2'):
    tfidf = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df, max_df=max_df,
        sublinear_tf=sublinear_tf,
        norm=norm
    )
    X_tr = tfidf.fit_transform(train_texts)
    X_te = tfidf.transform(test_texts)
    return X_tr, X_te, tfidf

def build_xgb(num_classes, learning_rate=0.1, max_depth=7, subsample=0.8,
              colsample_bytree=0.8, reg_lambda=3.0, n_estimators=2000,
              tree_method=TREE_METHOD):
    params = dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=reg_lambda,
        random_state=SEED,
        tree_method=tree_method
    )
    if num_classes == 2:
        clf = XGBClassifier(objective='binary:logistic', **params)
    else:
        clf = XGBClassifier(objective='multi:softprob', num_class=num_classes, **params)
    return clf


In [ ]:

# ---------- Variant B: Sentence Embeddings ----------
def embed_texts(model_name, texts, batch_size=64):
    model = SentenceTransformer(model_name)
    X = model.encode(list(texts), batch_size=batch_size,
                     show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)
    return X

def hstack_sparse_dense(X_sparse, X_dense):
    if not isinstance(X_sparse, csr_matrix):
        X_sparse = csr_matrix(X_sparse)
    X_dense_csr = csr_matrix(X_dense)
    return hstack([X_sparse, X_dense_csr]).tocsr()


In [ ]:

print("=== Running TF-IDF → XGBoost ===")
X_train_tfidf, X_test_tfidf, tfidf_vec = build_tfidf_features(
    X_train_txt, X_test_txt,
    ngram_range=(1,2),
    max_features=30000,
    min_df=5, max_df=0.9,
    sublinear_tf=True, norm='l2'
)

def build_model_A():
    return build_xgb(num_classes, learning_rate=0.1, max_depth=7,
                     subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0,
                     n_estimators=2000, tree_method=TREE_METHOD)

meanA, stdA, timeA = cv_run(X_train_tfidf, y_train, build_model_A, n_splits=N_SPLITS)

clfA = build_model_A()
clfA.fit(X_train_tfidf, y_train,
         eval_set=[(X_test_tfidf, y_test)],
         eval_metric='mlogloss' if num_classes > 2 else 'logloss',
         early_stopping_rounds=50, verbose=False)
testA = evaluate_clf(clfA, X_test_tfidf, y_test)
print("TF-IDF Test:", testA)


In [ ]:

print("\n=== Running Embeddings (all-MiniLM-L6-v2) → XGBoost ===")
emb_model = 'sentence-transformers/all-MiniLM-L6-v2'
start = time.time()
X_train_emb = embed_texts(emb_model, X_train_txt)
X_test_emb  = embed_texts(emb_model, X_test_txt)
embed_time = time.time() - start
print(f"Embedding time (s): {embed_time:.2f}, dims: {X_train_emb.shape[1]}")

def build_model_B():
    return build_xgb(num_classes, learning_rate=0.1, max_depth=10,
                     subsample=0.8, colsample_bytree=0.9, reg_lambda=2.0,
                     n_estimators=2000, tree_method=TREE_METHOD)

meanB, stdB, timeB = cv_run(X_train_emb, y_train, build_model_B, n_splits=N_SPLITS)

clfB = build_model_B()
clfB.fit(X_train_emb, y_train,
         eval_set=[(X_test_emb, y_test)],
         eval_metric='mlogloss' if num_classes > 2 else 'logloss',
         early_stopping_rounds=50, verbose=False)
testB = evaluate_clf(clfB, X_test_emb, y_test)
print("Embeddings Test:", testB)


In [ ]:

print("\n=== Running Hybrid (TF-IDF ⊕ Embeddings) → XGBoost ===")
X_train_h = hstack_sparse_dense(X_train_tfidf, X_train_emb)
X_test_h  = hstack_sparse_dense(X_test_tfidf, X_test_emb)

def build_model_C():
    return build_xgb(num_classes, learning_rate=0.1, max_depth=9,
                     subsample=0.8, colsample_bytree=0.9, reg_lambda=3.0,
                     n_estimators=2000, tree_method=TREE_METHOD)

meanC, stdC, timeC = cv_run(X_train_h, y_train, build_model_C, n_splits=N_SPLITS)

clfC = build_model_C()
clfC.fit(X_train_h, y_train,
         eval_set=[(X_test_h, y_test)],
         eval_metric='mlogloss' if num_classes > 2 else 'logloss',
         early_stopping_rounds=50, verbose=False)
testC = evaluate_clf(clfC, X_test_h, y_test)
print("Hybrid Test:", testC)


In [ ]:

results = pd.DataFrame([
    {'variant': 'TFIDF',
     'cv_macro_f1_mean': meanA['macro_f1'], 'cv_macro_f1_std': stdA['macro_f1'],
     'test_macro_f1': testA['macro_f1'], 'test_accuracy': testA['accuracy'], 'test_roc_auc': testA['roc_auc'],
     'cv_time_s': timeA},
    {'variant': 'EMB',
     'cv_macro_f1_mean': meanB['macro_f1'], 'cv_macro_f1_std': stdB['macro_f1'],
     'test_macro_f1': testB['macro_f1'], 'test_accuracy': testB['accuracy'], 'test_roc_auc': testB['roc_auc'],
     'cv_time_s': timeB},
    {'variant': 'HYBRID',
     'cv_macro_f1_mean': meanC['macro_f1'], 'cv_macro_f1_std': stdC['macro_f1'],
     'test_macro_f1': testC['macro_f1'], 'test_accuracy': testC['accuracy'], 'test_roc_auc': testC['roc_auc'],
     'cv_time_s': timeC},
]).sort_values('test_macro_f1', ascending=False).reset_index(drop=True)

results


In [ ]:

best = results.iloc[0]['variant']
print("Best variant:", best)
if best == 'TFIDF':
    y_pred = clfA.predict(X_test_tfidf)
elif best == 'EMB':
    y_pred = clfB.predict(X_test_emb)
else:
    y_pred = clfC.predict(X_test_h)

print("\nClassification report on test:")
print(classification_report(y_test, y_pred, target_names=target_names))
